# 10_Gold_Encounter

Gold-layer notebook. Attach `FHIR_Lakehouse` before running.

In [1]:
from pyspark.sql.functions import (
    col, element_at, regexp_extract, to_timestamp,
    unix_timestamp, when, current_timestamp
)
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

StatementMeta(, ea5bd2ca-0a1a-405f-a494-0f8010224983, 3, Finished, Available, Finished, False)

DataFrame[]

In [2]:
silver_encounter_df = spark.table("silver.encounter")
print("Silver Encounter Records:", silver_encounter_df.count())

StatementMeta(, ea5bd2ca-0a1a-405f-a494-0f8010224983, 4, Finished, Available, Finished, False)

Silver Encounter Records: 2005


In [3]:
gold_encounter_df = silver_encounter_df.select(
    col("id").alias("encounter_id"),
    col("subject.reference").alias("patient_reference"),
    regexp_extract(col("subject.reference"), r"Patient/([^/]+)", 1).alias("patient_id"),
    col("status").alias("encounter_status"),
    col("class.code").alias("encounter_class_code"),
    col("class.display").alias("encounter_class"),
    element_at(col("type"), 1).getField("text").alias("encounter_type"),
    to_timestamp(col("period.start")).alias("start_datetime"),
    to_timestamp(col("period.end")).alias("end_datetime"),
    col("serviceProvider.display").alias("service_provider"),
    col("meta.lastUpdated").alias("source_last_updated"),
    col("extraction_timestamp"),
    col("load_timestamp")
).withColumn(
    "duration_minutes",
    when(col("start_datetime").isNotNull() & col("end_datetime").isNotNull(),
         (unix_timestamp("end_datetime") - unix_timestamp("start_datetime")) / 60)
).withColumn("gold_load_timestamp", current_timestamp())

StatementMeta(, ea5bd2ca-0a1a-405f-a494-0f8010224983, 5, Finished, Available, Finished, False)

In [4]:
(gold_encounter_df.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true").saveAsTable("gold.fact_encounter"))

StatementMeta(, ea5bd2ca-0a1a-405f-a494-0f8010224983, 6, Finished, Available, Finished, False)

In [5]:
spark.table("gold.fact_encounter").printSchema()
spark.table("gold.fact_encounter").show(10, truncate=False)
print("Total Gold Encounter Records:", spark.table("gold.fact_encounter").count())

StatementMeta(, ea5bd2ca-0a1a-405f-a494-0f8010224983, 7, Finished, Available, Finished, False)

root
 |-- encounter_id: string (nullable = true)
 |-- patient_reference: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- encounter_status: string (nullable = true)
 |-- encounter_class_code: string (nullable = true)
 |-- encounter_class: string (nullable = true)
 |-- encounter_type: string (nullable = true)
 |-- start_datetime: timestamp (nullable = true)
 |-- end_datetime: timestamp (nullable = true)
 |-- service_provider: string (nullable = true)
 |-- source_last_updated: string (nullable = true)
 |-- extraction_timestamp: timestamp (nullable = true)
 |-- load_timestamp: timestamp (nullable = true)
 |-- duration_minutes: double (nullable = true)
 |-- gold_load_timestamp: timestamp (nullable = true)

+------------------------------------+--------------------------------------------+------------------------------------+----------------+--------------------+---------------+------------------------------------------+-------------------+-------------------+--------